# 03 - Split Project, Provider, and Customer Views

This notebook demonstrates the first version of the allocation layer. The goal is to avoid mixing overall project feasibility, Provider monetization, and customer value in one dashboard metric.


In [ ]:
from pathlib import Path
import sys

import pandas as pd

ROOT = Path.cwd().parents[0] if Path.cwd().name == 'notebooks' else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

from src.financial_model import FeasibilityInputs, MachineSpec, load_machine_specs, summarize_feasibility
from src.view_model import ViewAssumptions, build_views, views_to_records


## Load Machine Specs And View Presets


In [ ]:
machine_specs = load_machine_specs(ROOT / 'data' / 'processed' / 'machine_specs.csv')
view_presets = pd.read_csv(ROOT / 'data' / 'processed' / 'view_assumptions.csv')
view_presets


## Example: OpEx Rental Scenario

This uses the same broad structure as the Excel OpEx benchmark, then allocates value between Provider and the customer.


In [ ]:
def machine_by_capacity(capacity_ton: float) -> MachineSpec:
    row = machine_specs.loc[machine_specs['capacity_ton'].eq(capacity_ton)].iloc[0]
    return MachineSpec(
        model=str(row['model']),
        capacity_ton=float(row['capacity_ton']),
        total_load_kw=float(row['total_load_kw']),
        operation_hours_per_day=float(row['operation_hours_per_day']),
        asia_price_no_ce_usd=float(row['asia_price_no_ce_usd']),
        monthly_rental_ratio=float(row['monthly_rental_ratio']),
    )

inputs = FeasibilityInputs(
    business_model='opex',
    waste_tons_per_day=100,
    collection_rate=1,
    days_per_month=30,
    conversion_rate=0.7,
    fertilizer_price_usd_per_ton=250,
    service_fee_usd_per_ton=0,
    electricity_rate_usd_per_kwh=0.2033,
    electricity_load_factor=0.34431874077717656,
    enzyme_cost_usd_per_ton_waste=0,
    direct_labor_cost_usd_per_month=0,
    maintenance_cost_usd_per_month=0,
    tax_rate=0.10,
)

preset = view_presets.loc[view_presets['scenario'].eq('opex_provider_rental')].iloc[0]
assumptions = ViewAssumptions(
    current_disposal_fee_usd_per_ton=float(preset['current_disposal_fee_usd_per_ton']),
    provider_fertilizer_revenue_share=float(preset['provider_fertilizer_revenue_share']),
    provider_service_fee_share=float(preset['provider_service_fee_share']),
    provider_rental_revenue_share=float(preset['provider_rental_revenue_share']),
    provider_cogs_share=float(preset['provider_cogs_share']),
    provider_non_rental_opex_share=float(preset['provider_non_rental_opex_share']),
    provider_non_operating_expense_share=float(preset['provider_non_operating_expense_share']),
    customer_initial_investment_share=float(preset['customer_initial_investment_share']),
)

views = build_views(inputs, machine_by_capacity(60), assumptions)
pd.DataFrame(views_to_records(views))


## Interpretation

- The project view answers whether the opportunity works overall.
- The Provider view answers what Provider captures under the selected allocation assumptions.
- The customer view answers whether avoided disposal cost and any retained upside justify the cost.

These assumptions should become dashboard presets only after the business model is confirmed with Provider.
